# Lab 5, Day 2 — Pipeline, Features, and Model Selection

Build a leak-free `Pipeline`, get a cross-validated baseline, engineer features with a
stated hypothesis, compare models honestly, tune once, and evaluate on the test set
exactly once. See `Lab5_Day2_Instructions.md` for the full walkthrough.

This continues directly from Day 1's folder and split - not a restart.

## Before you start: watch leakage happen

Fit a `StandardScaler` on your *full* dataset (`X`, before any split) and print
`scaler.mean_`. Then fit a fresh one on `X_train` alone and print its `.mean_`. The
numbers differ. Before reading further, think about what that difference actually
means - which numbers were influenced by data your model should never have seen at
fit time? That's the entire argument for wrapping every fitted step in a `Pipeline`,
which is what you're about to build.

In [1]:
# TODO: the leakage demo described above (optional to keep in your final notebook,
# but do it before writing any pipeline code)
import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler

split = joblib.load("split.joblib")
X_train = split["X_train"]
X_test = split["X_test"]
X = pd.concat([X_train, X_test])

# Use complete numeric columns so the scaler can be fitted for this demonstration.
leakage_cols = ["Pclass", "SibSp", "Parch", "Fare"]
print("Full-data means:", StandardScaler().fit(X[leakage_cols]).mean_)
print("Training-only means:", StandardScaler().fit(X_train[leakage_cols]).mean_)


Full-data means: [ 2.29488159  0.49885409  0.38502674 33.29547928]
Training-only means: [ 2.30563515  0.4851958   0.40019102 34.0556522 ]


In [2]:
import pandas as pd
import numpy as np
import joblib

# TODO: reload yesterday's split with joblib.load("split.joblib"), or re-run Day 1's
# Step 5-6 if you didn't save it
from pipeline import build_preprocessor, build_pipeline, engineer, get_column_groups

split = joblib.load("split.joblib")
X_train = split["X_train"]
X_test = split["X_test"]
y_train = split["y_train"]
y_test = split["y_test"]


## Step 1: The ColumnTransformer (`pipeline.py`)

In [3]:
# TODO: build_preprocessor(num_cols, cat_cols) - a numeric sub-pipeline (impute then
# scale) and a categorical sub-pipeline (impute then one-hot encode). Remember
# handle_unknown on the encoder - a category seen only at predict time must not crash.
num_cols, cat_cols = get_column_groups(X_train)
preprocessor = build_preprocessor(num_cols, cat_cols)
preprocessor


ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Age', 'SibSp', 'Parch', 'Fare']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('encoder',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked',
                                  'home.dest', 'Pclass'])])

In [4]:
# TODO: build_pipeline(num_cols, cat_cols, model) - pre + model, ready to fit
from sklearn.linear_model import LogisticRegression

baseline_pipeline = build_pipeline(
    num_cols, cat_cols, LogisticRegression(max_iter=1000, random_state=0)
)


## Step 2: Cross-validated baseline

F1 is used because survival is imbalanced, so accuracy alone could hide poor survival predictions.

In [5]:
# TODO: cross_val_score on the training set only, an appropriate metric for this
# target's class balance, and report BOTH the mean and the standard deviation
from sklearn.model_selection import StratifiedKFold, cross_val_score

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
baseline_scores = cross_val_score(
    baseline_pipeline, X_train, y_train, cv=cv, scoring="f1"
)
print(f"Baseline F1: {baseline_scores.mean():.3f} +/- {baseline_scores.std():.3f}")


Baseline F1: 0.728 +/- 0.036


## Step 3: Engineer features, with a stated hypothesis first

In [6]:
# TODO: engineer(df) in pipeline.py - for each feature, write the hypothesis as a
# comment before the code. Then actually test whether it helped the CV score, and
# report the result either way, even if it didn't help.
# Hypothesis: family size, cabin availability, and title each add useful survival information.
drop_cols = ["Name", "Ticket", "Cabin", "home.dest"]
X_train_basic = X_train.drop(columns=drop_cols, errors="ignore")

X_train_family = X_train_basic.copy()
X_train_family["family_size"] = X_train["SibSp"] + X_train["Parch"] + 1
X_train_family["is_alone"] = (X_train_family["family_size"] == 1).astype(int)

X_train_cabin = X_train_basic.copy()
X_train_cabin["has_cabin"] = X_train["Cabin"].notna().astype(int)

X_train_title = X_train_basic.copy()
titles = X_train["Name"].str.extract(r",\s*([^\.]+)\.", expand=False)
X_train_title["Title"] = titles.where(
    titles.isin(["Mr", "Miss", "Mrs", "Master"]), "Other"
)

X_train_engineered = engineer(X_train)

basic_num, basic_cat = get_column_groups(X_train_basic)
basic_model = build_pipeline(
    basic_num, basic_cat, LogisticRegression(max_iter=1000, random_state=0)
)
basic_scores = cross_val_score(basic_model, X_train_basic, y_train, cv=cv, scoring="f1")
print(f"Without engineered features: {basic_scores.mean():.3f} +/- {basic_scores.std():.3f}")

feature_sets = {
    "Family features": X_train_family,
    "Cabin feature": X_train_cabin,
    "Title feature": X_train_title,
    "All engineered features": X_train_engineered,
}
for name, features in feature_sets.items():
    num_cols, cat_cols = get_column_groups(features)
    model = build_pipeline(
        num_cols, cat_cols, LogisticRegression(max_iter=1000, random_state=0)
    )
    scores = cross_val_score(model, features, y_train, cv=cv, scoring="f1")
    print(f"{name}: {scores.mean():.3f} +/- {scores.std():.3f}")

engineered_num, engineered_cat = get_column_groups(X_train_engineered)


Without engineered features: 0.707 +/- 0.033
Family features: 0.712 +/- 0.026
Cabin feature: 0.712 +/- 0.033
Title feature: 0.746 +/- 0.029
All engineered features: 0.754 +/- 0.029


## Step 4: Compare at least three models

In [7]:
# TODO: cross-validate at least three different model types with the same
# preprocessing, and report mean + std for each. Are the differences bigger than the
# fold-to-fold noise?
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

models = {
    "Logistic regression": LogisticRegression(max_iter=1000, random_state=0),
    "Random forest": RandomForestClassifier(n_estimators=200, random_state=0),
    "Gradient boosting": GradientBoostingClassifier(random_state=0),
}
model_scores = {}
for name, model in models.items():
    pipeline = build_pipeline(engineered_num, engineered_cat, model)
    scores = cross_val_score(pipeline, X_train_engineered, y_train, cv=cv, scoring="f1")
    model_scores[name] = scores
    print(f"{name}: {scores.mean():.3f} +/- {scores.std():.3f}")

best_name = max(model_scores, key=lambda name: model_scores[name].mean())
print(f"Highest mean F1: {best_name}. Compare the mean gaps with the standard deviations.")


Logistic regression: 0.754 +/- 0.029
Random forest: 0.731 +/- 0.035
Gradient boosting: 0.736 +/- 0.047
Highest mean F1: Logistic regression. Compare the mean gaps with the standard deviations.


## Step 5: Tune the best model, then evaluate the test set exactly once

In [8]:
# TODO: GridSearchCV on training data only (remember the model__param prefix for
# a parameter inside a named pipeline step)
from sklearn.model_selection import GridSearchCV

param_grids = {
    "Logistic regression": {"model__C": [0.1, 1, 10]},
    "Random forest": {"model__n_estimators": [100, 200], "model__max_depth": [None, 5]},
    "Gradient boosting": {"model__n_estimators": [50, 100], "model__learning_rate": [0.05, 0.1]},
}
best_pipeline = build_pipeline(engineered_num, engineered_cat, models[best_name])
grid = GridSearchCV(best_pipeline, param_grids[best_name], cv=cv, scoring="f1", n_jobs=-1)
grid.fit(X_train_engineered, y_train)
print("Best parameters:", grid.best_params_)
print(f"Best cross-validated F1: {grid.best_score_:.3f}")


Best parameters: {'model__C': 10}
Best cross-validated F1: 0.757


In [9]:
# TODO: the test set, touched here for the first and only time - report an
# appropriate metric. If the tuned model does no better than the baseline, that is a
# result to report, not a bug to hide.
from sklearn.metrics import f1_score

X_test_engineered = engineer(X_test)
test_predictions = grid.predict(X_test_engineered)
test_f1 = f1_score(y_test, test_predictions)
print(f"Final held-out test F1: {test_f1:.3f}")


Final held-out test F1: 0.758


## Closing analysis

The feature-comparison cell reports whether family, cabin, and title features helped
relative to the baseline; this result should be reported honestly after running the
notebook. In the previous run, all engineered features together improved F1 from 0.707
to 0.754. Logistic regression had the highest mean F1 (0.754), but its advantage over
the other models was similar to the fold-to-fold variation, so there is no clear winner.
After tuning, the held-out test F1 was 0.758. With more time, I would try fare-by-class
and more detailed title features.